# curious-george — Phase 0 validation notebook

Runs the already-validated foundation (memory store, fabricated ground truth, loss measurement, LoRA fine-tuning) against GPU compute, then moves on to Phase 1 experiments. See the repo [README](https://github.com/stvenmobile/curious-george#readme) for the full roadmap and the definitions behind each term used below.

In [ ]:
!git clone https://github.com/stvenmobile/curious-george.git
%cd curious-george
!pip install -q -r requirements.txt

In [ ]:
import sys
sys.path.insert(0, "src")

from curious_george.warble_harness import run_sanity_check
run_sanity_check(device="cuda")

## Persisting memory across sessions

Colab's local disk is wiped on every runtime restart. `MemoryStore`'s whole reason for existing is a `loss_history` that survives across sessions — without that, `learning_progress()` never has more than one measurement to compare against, and the "learn and retain" premise of the project doesn't hold.

This cell mounts Google Drive and points `MemoryStore.save()`/`load()` at a folder there via the `CURIOUS_GEORGE_MEMORY_DIR` environment variable, instead of the ephemeral local `data/memory/` default. Run it once per session, before any `save()`/`load()` call with no explicit path — it's read at call time, so it doesn't need to happen in the same cell as those calls.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ["CURIOUS_GEORGE_MEMORY_DIR"] = "/content/drive/MyDrive/curious-george-memory"

from curious_george.memory_store import MemoryStore
store = MemoryStore.load()   # picks up last session's state from Drive, or starts empty on first run
print(f"Loaded {len(store)} item(s) from persistent memory.")

## The genuine learning test: LoRA fine-tuning, cold measurement

The sanity check above proved the plumbing works, but its "study" step just placed warble facts directly in the prompt — a confound, since *any* coherent context measurably helps next-token prediction (which is exactly why quaddles, whose own facts were never shown, still improved: +1.15 nats of that improvement is just reading-comprehension priming, not learning).

`run_lora_check` fixes this. It fine-tunes a small LoRA adapter (the base model's own weights are frozen and never change) on warble facts only, then measures loss on **cold** prompts — no warble content anywhere in the prompt, before or after fine-tuning. Any improvement can only come from the adapter's trained weights, not from priming. The original CPU run (`piper_assistant`, pre-port) took ~41 minutes and found: warbles improved 2.65 nats cold, quaddles (never trained on, structurally similar) leaked 0.68 nats (a ~26% residual confound from shared sentence structure between the two fabricated species — see the README's Phase 4). This run should reproduce that on GPU, much faster.

In [ ]:
from curious_george.warble_harness import run_lora_check
run_lora_check(device="cuda")

### Result (2026-09-12, T4 GPU)

| | before | after | delta | accuracy |
|---|---|---|---|---|
| warbles (LoRA-trained) | 3.4684 | 0.7177 | **+2.7507** | 0.483 -> 0.913 |
| quaddles (never trained on) | 3.6189 | 3.0267 | **+0.5922** | 0.271 -> 0.435 |

Leak ratio 0.215 — reproduces the original CPU run's finding (2.65 / 0.68 nats, ratio 0.258) almost exactly, on different hardware, with accuracy landing on the identical 0.913 / 0.435 both times. Confirms the port is sound and the result isn't hardware-specific.

Same interpretation as before: genuine, persistent, cold, mostly fact-specific learning from the LoRA adapter's own weights — real evidence for the core mechanism — with a real, if smaller, residual leak from warbles and quaddles sharing sentence structure. That leak is the open question Phase 4 in the README exists to address; Phase 1 (defining and testing a curiosity score) is the more immediate next step.

## Phase 1: does baseline loss category predict actual learning progress?

Phase 0's `recall_probes` were always decompositions of the *exact* sentences trained on — e.g. `study_facts` has "Warbles are green and yellow mammals." and the matching probe is that same sentence split into a prompt and target. That measures memorization: does the model recall the specific thing it was shown. It has never measured **generalization**: does studying some facts about a topic make the model any better at completing *different* material about that same topic.

That distinction is the actual definition of "noise" this project is using (see the README): a topic has exploitable structure if studying it transfers to unseen material; it's noise if the model can only memorize exactly what it saw, with nothing carrying over.

**This section went through two failed designs before landing on the one below — both failures are real findings, not just bugs, and are recorded here rather than edited out.** See "What didn't work" further down for the full story. The short version: generalization can't be measured by withholding some of a topic's *own* facts from training, because a small-capacity LoRA adapter trained on several completions of the same prefix (e.g. "Warbles...") builds a near-deterministic mapping from that prefix to those specific completions — a withheld fact sharing the same prefix collides with that mapping and gets *actively worse*, regardless of how much real structure the topic has. That's a genuine phenomenon (interference), just a different one than what this section is trying to measure.

The design that actually isolates generalization: pair each topic with a **separate sibling topic** — same template/register, different subject and vocabulary, never trained on at all. This is exactly how Phase 0 already measured the warble→quaddle leak, just formalized across three anchor categories:

- **known** — real-world common-knowledge sentences, paired with a second, unrelated set of common-knowledge sentences.
- **moderate** — Phase 0's original 10 warble facts, paired with quaddles (the same pair Phase 0 already validated).
- **noise** — word-salad sentences, paired with a second word-salad set drawn from a completely disjoint 64-word vocabulary (verified programmatically — zero shared words with the trained set).

`run_topic_trial` measures loss on both the topic's own `trained_probes` (memorization) and its sibling's probes (generalization) before and after a LoRA study step (200 steps, 1e-4 — Phase 0's validated regime) on `train_facts` only. `generalization_progress / memorization_progress` is the noise measurement: near 0 means nothing transferred to the sibling; positive means it did.

No result recorded yet below — this rebuild hasn't been run for real yet.

In [ ]:
from curious_george.curiosity import run_noise_experiment
results = run_noise_experiment(device="cuda")

### What didn't work (kept for the record, not edited out)

**Attempt 1 — held-out facts, shared vocabulary noise content.** `noise`'s word-salad sentences reused ~25 words across all 10 sentences. Result: `noise`'s generalization ratio (0.420) came out *higher* than `moderate`'s (0.121) — backwards from the hypothesis. Cause: training on the trained noise sentences raised the model's probability on words the held-out sentences happened to share, which is real loss improvement but from vocabulary overlap, not structure.

**Attempt 2 — held-out facts, vocabulary leak fixed (80 distinct words, zero reuse).** All three categories came back *negative* this time — including `moderate`:

| topic | memorization | generalization | ratio |
|---|---|---|---|
| known | 0.6243 | -0.3197 | -0.512 |
| moderate | 2.7042 | -0.4144 | -0.153 |
| noise | 6.5294 | -0.4635 | -0.071 |

Fixing the noise leak didn't fix the real problem — it just exposed the next one. A controlled follow-up at Phase 0's own validated hyperparameters (200 steps, 1e-4, ruling out "too aggressive a cheap-trial learning rate" as the cause) made it *worse*, not better: `moderate`'s held-out generalization dropped to **-2.2741** (loss went from 3.65 to 5.92 — much worse than before training). Meanwhile the exact same training run improved Phase 0's *quaddle* probes by **+0.7134** — nearly matching Phase 0's original +0.68.

Same model, same training run, opposite signs, depending only on whether the probe shares the trained prefix. Mechanism: training on 8 different completions of `"Warbles..."` teaches the adapter a narrow, near-deterministic mapping from that prefix to those specific completions. A held-out fact sharing the same prefix collides with that mapping and gets actively worse — that's interference, not a measurement of whether the topic has learnable structure. A probe with a different subject (quaddles) never triggers the collision, so the positive number there is the genuine structural leak Phase 0 already found. Phase 0 never surfaced this because `run_lora_check` always trained on the *full* set of 10 warble facts together — nothing sharing that prefix was ever held back to collide with.

This is why the design above uses a separate sibling topic instead of same-topic held-out facts.